In [ ]:
import pandas as pd 
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pingouin as pg
import matplotlib
from glob import glob
import os
import sys
sys.path.extend(['/Users/amonast/Documents/GitHub/zlab_miniscope'])
from load_traces import load_aligned_act,build_intervals
from scipy.ndimage import gaussian_filter1d
from mpl_toolkits.mplot3d import Axes3D
from itertools import product
%matplotlib widget
# mixed color
matplotlib.rcParams['svg.fonttype']='none'
traces_dir = '/Users/amonast/Desktop/miniscope/Traces'
behavior_csv = '/Users/amonast/Desktop/miniscope/data/tuning/intervals_allmice.csv'
data_info_csv = '/Users/amonast/Desktop/miniscope/dbnst_data_info.csv'

In [ ]:
animals =   ['C1F2','C1M1','C1M2', 'C2F1','C2M4', 'C3F3', 'C3F4', 
                'C3M2','C3M3', 'C3M4','C0F2C']

In [ ]:
def get_tuned_df(animals,data_dir,sessions=['R1','R2']):
    index_cols = ['animal', 'session', 'behavior_type']
    rows = []
    for ani, session in product(animals, sessions):
        pf = f"{data_dir}/tuning/p_values_{ani}_{session}.csv"
        zf = f"{data_dir}/tuning/z_scores_{ani}_{session}.csv"
        if not (os.path.exists(pf) and os.path.exists(zf)):
            print(f"skipping {ani} {session}: no tuning files")
            continue
        df_p = pd.read_csv(pf).set_index(index_cols)
        df_z = pd.read_csv(zf).set_index(index_cols)

        chunk = pd.concat({'pval': df_p, 'zscore': df_z}, names=['metric'])
        chunk = chunk.reorder_levels(['animal', 'session', 'behavior_type', 'metric'])
        rows.append(chunk)
    df = pd.concat(rows).sort_index()

    sig = df.xs('pval', level='metric').le(0.05)
    pos = df.xs('zscore', level='metric').gt(0)
    tuned = ((sig & pos).apply(lambda row: row.index[row].tolist(), axis=1)
                .rename('tuned_cells')
                .reset_index())

    ## exlcude these animals cells as aggessive tuned b/c one was the intruder mouse and one only had one tiny bout 
    tuned = tuned[~((tuned["animal"].isin( ["C2F1", "C2M4", "C1M2"]  )) & (tuned["behavior_type"] == "Aggressive"))]
    return tuned,df

In [ ]:
tuned,df = get_tuned_df(animals = animals,
                     data_dir='/Users/amonast/Desktop/miniscope/data')

In [ ]:
df

In [ ]:
results = {}

for animal, group in df.groupby(level='animal'):
    pval_rows = group.xs('pval', level='metric')
    zscore_rows = group.xs('zscore', level='metric')
    
    animal_results = {}
    for behavior, beh_group in pval_rows.groupby(level='behavior_type'):
        stacked_pval = beh_group.stack()
        stacked_pval.index.set_names('cell', level=-1, inplace=True)
        sig = stacked_pval[stacked_pval < 0.05].sort_values()

        zscore_beh = zscore_rows.xs(behavior, level='behavior_type')
        stacked_zscore = zscore_beh.stack()
        stacked_zscore.index.set_names('cell', level=-1, inplace=True)

        # sig_df has (session, cell, pval); zscore_df has (session, cell, zscore)
        sig_df = sig.rename('pval').reset_index()[['session', 'cell', 'pval']]
        zscore_df = stacked_zscore.rename('zscore').reset_index()[['session', 'cell', 'zscore']]
        merged = sig_df.merge(zscore_df, on=['session', 'cell'], how='left')
        merged['animal'] = animal
        merged['behavior_type'] = behavior
        merged = merged.set_index(['animal', 'session', 'behavior_type', 'cell'])

        animal_results[behavior] = {
            'pvals': merged['pval'],
            'zscores': merged['zscore'],
        }
    
    results[animal] = animal_results

In [ ]:
from load_traces import load_aligned_act

In [ ]:
session_cache = {}  # (animal, session) -> loaded act DataFrame
cell_traces = {}    # (animal, session, behavior, cell) -> {activity, behavior_intervals, pval, zscore}

for animal, animal_dict in results.items():
    for behavior, beh_dict in animal_dict.items():
        for (animal, session, behavior_type, cell) in beh_dict['pvals'].index:
            key = (animal, session)
            if key not in session_cache:
                session_cache[key] = load_aligned_act(
                    animal=animal,
                    session=session,
                    traces_dir=traces_dir,
                    behavior_csv=behavior_csv,
                    info_csv=data_info_csv,)
            
            act = session_cache[key]
            cell_traces[(animal, session, behavior_type, cell)] = {
                "activity": act[cell],
                "behavior_intervals": act[act["behavior_type"] == behavior_type][[cell, "Behavior", "behavior_type"]],
                "pval": beh_dict['pvals'].loc[(animal, session, behavior_type, cell)],
                "zscore": beh_dict['zscores'].loc[(animal, session, behavior_type, cell)],
            }

In [ ]:
cell_traces

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


In [ ]:
BEHAVIOR_COLORS = {
    'Aggressive': 'red',
    'Pro-Social': 'royalblue',
    'Social Avoidance': 'purple',
    'Baseline': 'gray',
}

def plot_cell_trace(full_trace, act, title):
    time = full_trace.index.get_level_values('timestamp').total_seconds()
    
    fig, ax = plt.subplots(figsize=(14, 3))
    smoothed = gaussian_filter1d(full_trace,sigma=3)
    ax.plot(time, smoothed, lw=1, color='k')
    
    patches = []
    for behavior, color in BEHAVIOR_COLORS.items():
        intervals = act[act["behavior_type"] == behavior]
        if intervals.empty:
            continue
        in_behavior = intervals.index.get_level_values('timestamp').total_seconds()
        breaks = np.where(np.diff(in_behavior) > 0.1)[0] + 1
        chunks = np.split(in_behavior, breaks)
        for chunk in chunks:
            ax.axvspan(chunk[0], chunk[-1], alpha=0.2, color=color, lw=0)
        patches.append(mpatches.Patch(color=color, alpha=0.2, label=behavior))
    
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Z-score')
    ax.set_title(title)
    ax.legend(handles=patches, loc='upper right')
    plt.tight_layout()
    plt.show()


In [ ]:
[k[3] for k in cell_traces if k[:3] == ('C3M2', 'R2', 'Aggressive')]

In [ ]:
%matplotlib widget
import seaborn as sb

In [ ]:
ani,session,cell='C1F2','R2','14'
e = cell_traces[(ani,session, 'Pro-Social',cell)]
plot_cell_trace(e['activity'], session_cache[(ani, session)], 
                title=f'{ani} {session} | Cell {cell} | zscore={e["zscore"]:.3f}| p={e["pval"]:.3f}')
sb.despine()

In [ ]:
tuned.to_csv('/Users/amonast/Desktop/miniscope/data/tuned_dataframe.csv')

In [ ]:
# from matplotlib.backends.backend_pdf import PdfPages

# output_dir = '/Users/amonast/Desktop/miniscope/figures'
# os.makedirs(output_dir, exist_ok=True)

# # Group cell_traces keys by (animal, session)
# from collections import defaultdict
# by_session = defaultdict(list)
# for key in cell_traces:
#     animal, session, behavior, cell = key
#     by_session[(animal, session)].append(key)

# for (animal, session), keys in by_session.items():
#     pdf_path = f"{output_dir}/{animal}_{session}_tuned_cells.pdf"
#     with PdfPages(pdf_path) as pdf:
#         for key in keys:
#             animal, session, behavior, cell = key
#             e = cell_traces[key]
#             fig, ax = plt.subplots(figsize=(14, 3))
            
#             # reuse plot logic inline so we can pass ax
#             time = e['activity'].index.get_level_values('timestamp').total_seconds()
#             smoothed = gaussian_filter1d(e['activity'], sigma=3)
#             ax.plot(time, smoothed, lw=1, color='k')
            
#             act = session_cache[(animal, session)]
#             patches = []
#             for beh, color in BEHAVIOR_COLORS.items():
#                 intervals = act[act["behavior_type"] == beh]
#                 if intervals.empty:
#                     continue
#                 in_beh = intervals.index.get_level_values('timestamp').total_seconds()
#                 breaks = np.where(np.diff(in_beh) > 0.1)[0] + 1
#                 chunks = np.split(in_beh, breaks)
#                 for chunk in chunks:
#                     ax.axvspan(chunk[0], chunk[-1], alpha=0.2, color=color, lw=0)
#                 patches.append(mpatches.Patch(color=color, alpha=0.2, label=beh))
            
#             ax.set_xlabel('Time (s)')
#             ax.set_ylabel('Z-score')
#             ax.set_title(f'{animal} {session} | Cell {cell} | {behavior} | zscore={e["zscore"]:.3f} | p={e["pval"]:.3f}')
#             ax.legend(handles=patches, loc='upper right')
#             sns.despine(ax=ax)
#             plt.tight_layout()
#             pdf.savefig(fig)
#             plt.close(fig)
#     print(f"Saved {pdf_path}")

# Good example Cells:

In [ ]:
example_dict = {'Avoidance': [('C3M2','R1','82'),
                            ('C3F4','R2','11'),
                            ('C1M2','R2','26'),],
                'Pro-Social':[('C3M2','R1','13'),
                            ('C2F1','R1','39'),
                            ('C2F1','R1','12'),],
                'Aggressive': [('C3M2','R2','47'),
                               ('C3M3','R1','42'),
                               ('C3M2','R2','35')]}

In [ ]:
WINDOW_LEN = 480  # seconds shown in every panel
BEHAVIOR_KEY_MAP = {
    'Avoidance': 'Social Avoidance',
    'Pro-Social': 'Pro-Social',
    'Aggressive': 'Aggressive',
}

WINDOW_PAD = 90  # seconds before/after behavior bouts to show

plt.rcParams['font.size'] = 7
YTICK_FS = 6   # y-tick label font size

for behavior_label, examples in example_dict.items():
    behavior_key = BEHAVIOR_KEY_MAP.get(behavior_label, behavior_label)
    n = len(examples)
    fig, axes = plt.subplots(n, 1, figsize=(1.7, .3 * n),
                             sharex=False, sharey=False,
                             squeeze=False)
    fig.subplots_adjust(hspace=0.05)

    t_starts, t_ends = [], []

    for i, (animal, session, cell) in enumerate(examples):
        ax = axes[i][0]
        key = (animal, session, behavior_key, cell)
        e = cell_traces[key]
        act = session_cache[(animal, session)]

        time = e['activity'].index.get_level_values('timestamp').total_seconds()
        smoothed = gaussian_filter1d(e['activity'], sigma=3)

        s_min, s_max = smoothed.min(), smoothed.max()
        smoothed_norm = (smoothed - s_min) / (s_max - s_min) if s_max > s_min else smoothed * 0

        target_intervals = act[act["behavior_type"] == behavior_key]
        if not target_intervals.empty:
            bout_times = target_intervals.index.get_level_values('timestamp').total_seconds()
            t_start = max(time[0], bout_times.min() - WINDOW_PAD)
        else:
            t_start = time[0]
        # fixed length; slide back if the session ends before the window does
        t_start = max(time[0], min(t_start, time[-1] - WINDOW_LEN))
        t_end   = t_start + WINDOW_LEN

        mask = (time >= t_start) & (time <= t_end)
        ax.plot(time[mask], smoothed_norm[mask], lw=0.3, color='k')

        for beh, color in BEHAVIOR_COLORS.items():
            intervals = act[act["behavior_type"] == beh]
            if intervals.empty:
                continue
            in_beh = intervals.index.get_level_values('timestamp').total_seconds()
            in_beh = in_beh[(in_beh >= t_start) & (in_beh <= t_end)]
            if len(in_beh) == 0:
                continue
            breaks = np.where(np.diff(in_beh) > 0.1)[0] + 1
            chunks = np.split(in_beh, breaks)
            for chunk in chunks:
                ax.axvspan(chunk[0], chunk[-1], alpha=0.25, color=color, lw=0)

        ax.set_ylim(0, 1.1)
        ax.set_yticks([0, 1])
        ax.set_yticklabels(['0', '1'], fontsize=YTICK_FS)
        ax.set_ylabel('')
        ax.tick_params(axis='x', labelsize=5)
        ax.tick_params(axis='y', labelsize=YTICK_FS)
        ax.set_xticks([])
        ax.set_yticks([])
        sns.despine(ax=ax, bottom=True)

    
    ax_last = axes[-1][0]
    bar_len = 30
    x0, x1 = ax_last.get_xlim()
    bar_x = x1 - bar_len - (x1 - x0) * 0.02
    ax_last.plot([bar_x, bar_x + bar_len], [-0.15, -0.15],
                 color='k', lw=1.5, clip_on=False,
                 transform=ax_last.get_xaxis_transform())
    # ax_last.text(bar_x + bar_len / 2, -0.25, f'{bar_len} s',
    #              ha='center', va='top', fontsize=6,
    #              transform=ax_last.get_xaxis_transform())

    ax.set_xlim(t_start, t_end) 
    plt.show()
    plt.savefig(f'figures/{behavior_label}_tunedcells.svg', transparent=True)


In [ ]:

patches = [mpatches.Patch(color=color, alpha=0.5, label=beh)
           for beh, color in BEHAVIOR_COLORS.items()]

fig, ax = plt.subplots(figsize=(4, 0.4))
ax.legend(handles=patches, loc='center', ncol=len(patches),
          fontsize=7, frameon=False)
ax.axis('off')
plt.tight_layout()
plt.show()
plt.savefig('figures/behavior_legend.svg', transparent=True)


In [ ]:
# ── Build mixed-tuned cell dict ───────────────────────────────────────────────
# mixed = cells tuned (p<0.05, z>0) to >1 behavior type in the same session
mixed_cells = {}
for (ani, ses), grp in tuned.groupby(['animal', 'session']):
    cell_to_btypes = {}
    for _, row in grp.iterrows():
        for cell in row['tuned_cells']:
            cell_to_btypes.setdefault(cell, set()).add(row['behavior_type'])
    mixed = [c for c, btypes in cell_to_btypes.items() if len(btypes) > 1]
    if mixed:
        mixed_cells[(ani, ses)] = mixed

for k, v in mixed_cells.items():
    print(f'{k}: {len(v)} mixed cells — {v}')


In [ ]:
# ── Plot mixed-tuned cell traces ──────────────────────────────────────────────
plt.rcParams['font.size'] = 7
YTICK_FS = 5   # y-tick label font size

# Define the subset to plot: {(animal, session): [cell, ...]}
mixed_example_dict = {
    ('C1F2', 'R1'): ['19'],
    ('C3M2', 'R2'): ['47'],
    ('C3M4', 'R2'): ['42'],
}

# flatten all entries into a single list of (ani, ses, cell)
all_examples = [(ani, ses, cell)
    for (ani, ses), cells in mixed_example_dict.items()
    for cell in cells]

n = len(all_examples)
fig, axes = plt.subplots(n, 1, figsize=(1.7, 0.3 * n),
                         sharex=True, sharey=False, squeeze=False)
fig.subplots_adjust(hspace=0.05)

t_starts, t_ends = [], []

for i, (ani, ses, cell) in enumerate(all_examples):
    ax = axes[i][0]
    act = session_cache[(ani, ses)]

    full_trace = act[cell]
    time = full_trace.index.get_level_values('timestamp').total_seconds()
    smoothed = gaussian_filter1d(full_trace.values.astype(float), sigma=3)
    s_min, s_max = smoothed.min(), smoothed.max()
    smoothed_norm = (smoothed - s_min) / (s_max - s_min) if s_max > s_min else np.zeros_like(smoothed)

    social = act[act['behavior_type'].isin(['Pro-Social', 'Social Avoidance', 'Aggressive'])]
    if not social.empty:
        bout_times = social.index.get_level_values('timestamp').total_seconds()
        t_start = max(time[0], bout_times.min() - WINDOW_PAD)
        t_end   = min(time[-1], bout_times.max() + WINDOW_PAD)
    else:
        t_start, t_end = time[0], time[-1]
    t_starts.append(t_start)
    t_ends.append(t_end)

    mask = (time >= t_start) & (time <= t_end)
    ax.plot(time[mask], smoothed_norm[mask], lw=0.3, color='k')

    for beh, color in BEHAVIOR_COLORS.items():
        beh_rows = act[act['behavior_type'] == beh]
        if beh_rows.empty:
            continue
        in_beh = beh_rows.index.get_level_values('timestamp').total_seconds()
        in_beh = in_beh[(in_beh >= t_start) & (in_beh <= t_end)]
        if len(in_beh) == 0:
            continue
        breaks = np.where(np.diff(in_beh) > 0.1)[0] + 1
        for chunk in np.split(in_beh, breaks):
            ax.axvspan(chunk[0], chunk[-1], alpha=0.25, color=color, lw=0)

    cell_btypes = sorted(
        row['behavior_type']
        for _, row in tuned[
            (tuned['animal'] == ani) & (tuned['session'] == ses)
        ].iterrows()
        if cell in row['tuned_cells']
    )
    ax.set_ylim(0, 1.1)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(['0', '1'], fontsize=YTICK_FS)
    ax.tick_params(axis='x', labelsize=5)
    ax.tick_params(axis='y', labelsize=YTICK_FS)
    ax.set_xticks([])
    ax.set_yticks([])

    sns.despine(ax=ax, bottom=True)

ax_last = axes[-1][0]
bar_len = 30
t_start_all, t_end_all = min(t_starts), max(t_ends)
bar_x = t_end_all - bar_len - (t_end_all - t_start_all) * 0.02
ax_last.plot([bar_x, bar_x + bar_len], [-0.15, -0.15],
             color='k', lw=1.5, clip_on=False,
             transform=ax_last.get_xaxis_transform())
# ax_last.text(bar_x + bar_len / 2, -0.25, f'{bar_len} s',
#              ha='center', va='top', fontsize=6,
#              transform=ax_last.get_xaxis_transform())

axes[0,0].set_xlim([120,600])
axes[1,0].set_xlim([120,600])
axes[2,0].set_xlim([120,600])

plt.savefig('figures/mixed_cells.svg', transparent=True)
plt.show()


# Other cells

In [ ]:
# ── Plot Other-tuned example cells ────────────────────────────────────────────
plt.rcParams['font.size'] = 7
YTICK_FS = 5# y-tick label font size
WINDOW_LEN = 480   # seconds shown in every panel

other_examples = [('C1M1','R1','35'),
                  ('C2F1','R1','41'),
                  ('C3M3','R1','5')]
# optional per-panel start (s); anything not listed starts at 120
other_starts = {}   # e.g. {('C3M3','R1','5'): 200}

n = len(other_examples)
fig, axes = plt.subplots(n, 1, figsize=(1.7, 0.3 * n),
                         sharex=False, sharey=False, squeeze=False)
fig.subplots_adjust(hspace=0.05)

for i, (ani, ses, cell) in enumerate(other_examples):
    ax = axes[i][0]
    act = session_cache[(ani, ses)]

    full_trace = act[cell]
    time = full_trace.index.get_level_values('timestamp').total_seconds()
    smoothed = gaussian_filter1d(full_trace.values.astype(float), sigma=3)
    s_min, s_max = smoothed.min(), smoothed.max()
    smoothed_norm = (smoothed - s_min) / (s_max - s_min) if s_max > s_min else np.zeros_like(smoothed)

    t_start = other_starts.get((ani, ses, cell), 120)
    t_start = max(time[0], min(t_start, time[-1] - WINDOW_LEN))
    t_end   = t_start + WINDOW_LEN

    mask = (time >= t_start) & (time <= t_end)
    ax.plot(time[mask], smoothed_norm[mask], lw=0.3, color='k')

    for beh, color in BEHAVIOR_COLORS.items():
        beh_rows = act[act['behavior_type'] == beh]
        if beh_rows.empty:
            continue
        in_beh = beh_rows.index.get_level_values('timestamp').total_seconds()
        in_beh = in_beh[(in_beh >= t_start) & (in_beh <= t_end)]
        if len(in_beh) == 0:
            continue
        breaks = np.where(np.diff(in_beh) > 0.1)[0] + 1
        for chunk in np.split(in_beh, breaks):
            ax.axvspan(chunk[0], chunk[-1], alpha=0.25, color=color, lw=0)

    ax.set_xlim(t_start, t_end)
    ax.set_ylim(0, 1.1)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(['0', '1'], fontsize=YTICK_FS)
    ax.tick_params(axis='x', labelsize=5)
    ax.tick_params(axis='y', labelsize=YTICK_FS)
    ax.set_xticks([])
    ax.set_yticks([])

    sns.despine(ax=ax, bottom=True)

ax_last = axes[-1][0]
bar_len = 30
x0, x1 = ax_last.get_xlim()
bar_x = x1 - bar_len - (x1 - x0) * 0.02
ax_last.plot([bar_x, bar_x + bar_len], [-0.15, -0.15],
             color='k', lw=1.5, clip_on=False,
             transform=ax_last.get_xaxis_transform())
# ax_last.text(bar_x + bar_len / 2, -0.25, f'{bar_len} s',
#              ha='center', va='top', fontsize=6,
#              transform=ax_last.get_xaxis_transform())

plt.savefig('figures/other_cells.svg', transparent=True)
plt.show()